# NB10 – Validation + leak audit (v2)

**Tiered ResNet-18 sniff test** (per generator + overall) plus structural checks.

| 1-epoch accuracy | Reading | Action |
|---|---|---|
| < 85% | Healthy | Proceed |
| 85–95% | Investigate | Check Grad-CAM / FFT |
| > 95% | Leak signal | Stop, run decision tree |

GPU **ON**. Internet ON.

In [1]:
import importlib.util, sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
_need = []
for _m, _p in [("huggingface_hub","huggingface_hub>=0.23"),
               ("pyarrow","pyarrow"), ("PIL","pillow"), ("datasets","datasets")]:
    if importlib.util.find_spec(_m) is None: _need.append(_p)
if _need: _pip(*_need); print("installed:", _need)
else: print("all base deps present")

all base deps present


In [2]:
REPO_ID = "Shanmuk4622/ai-detection-dataset-v2"
SAMPLE_PER_CLASS = 1000   # images per class for the sniff test (raise for stronger signal)
EPOCHS = 1

import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
MASTER_SEED = cfg.get("master_seed", 42)   # change master_seed in NB00 to reseed everything
print(f"library {C.PIPELINE_VERSION} loaded | MASTER_SEED={MASTER_SEED}")
import torch
assert torch.cuda.is_available(), "Turn the GPU ON for NB10."
print("device: cuda")

ai_detect_common.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

library 1.2 loaded | MASTER_SEED=42
device: cuda


In [3]:
import io, numpy as np, torch
from PIL import Image

def load_images(prefix, n):
    out = []
    for f in C.list_shards(REPO_ID, prefix, TOKEN):
        loc = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
        t   = C.pq.read_table(loc, columns=["image"])
        for b in t.column("image").to_pylist():
            im = Image.open(io.BytesIO(b)).convert("RGB").resize((224,224), Image.BILINEAR)
            out.append(np.asarray(im, dtype=np.float32) / 255.0)
            if len(out) >= n: return out
    return out

print("loading reals ...")
reals = load_images("real/", SAMPLE_PER_CLASS)
print("real images:", len(reals))

loading reals ...


real/real-70beac4d-00000.parquet:   0%|          | 0.00/222M [00:00<?, ?B/s]

real/real-70beac4d-00001.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

real images: 1000


In [4]:
import torch, numpy as np
import torch.nn as nn
from torchvision.models import resnet18
from sklearn.model_selection import train_test_split

MEAN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

def to_batch(arrs):
    return (torch.from_numpy(np.stack(arrs)).permute(0,3,1,2) - MEAN) / STD

def sniff(pos, neg, tag=""):
    n = min(len(pos), len(neg))
    if n < 50: print(f"  [{tag}] too few ({n}); skip"); return None
    X = pos[:n]+neg[:n]; y = [1]*n+[0]*n
    Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.2, random_state=MASTER_SEED, stratify=y)
    dev = "cuda"
    net = resnet18(weights="IMAGENET1K_V1"); net.fc = nn.Linear(net.fc.in_features,2); net.to(dev)
    opt = torch.optim.Adam(net.parameters(), lr=1e-4); lossf = nn.CrossEntropyLoss()
    Xtr_t, ytr_t = to_batch(Xtr).to(dev), torch.tensor(ytr).to(dev)
    net.train()
    for _ in range(EPOCHS):
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(Xtr_t), 64):
            idx = perm[i:i+64]; opt.zero_grad()
            lossf(net(Xtr_t[idx]), ytr_t[idx]).backward(); opt.step()
    net.eval()
    with torch.no_grad():
        pred = net(to_batch(Xte).to(dev)).argmax(1).cpu().numpy()
    acc = float((pred == np.array(yte)).mean())
    v = "healthy (<85)" if acc<0.85 else ("INVESTIGATE (85-95)" if acc<0.95 else "LEAK? (>95)")
    print(f"  [{tag}] accuracy: {acc:.3f}  →  {v}")
    return acc

print("sniff-test trainer ready.")

sniff-test trainer ready.


In [5]:
import itertools, json, tempfile, os, random
results = {}
for m in cfg["generators"]:
    ai = load_images(f"data/{m}/", SAMPLE_PER_CLASS)
    if not ai: print(f"[{m}] no images yet"); continue
    results[m] = sniff(ai, reals, tag=m)

per = max(1, SAMPLE_PER_CLASS // max(1, len(cfg["generators"])))
ai_mixed = list(itertools.chain.from_iterable(
    load_images(f"data/{m}/", per) for m in cfg["generators"]))[:SAMPLE_PER_CLASS]
results["__overall__"] = sniff(ai_mixed, reals, tag="overall")

# structural checks
problems = []
try:
    sp = C.hf_hub_download(REPO_ID, "splits.parquet", repo_type="dataset", token=TOKEN)
    from collections import Counter
    spt = C.pq.read_table(sp)
    multi = [k for k,v in Counter(spt.column("source_real_id").to_pylist()).items() if v>1]
    if multi: problems.append(f"{len(multi)} source_real_id(s) in multiple splits")
except Exception as e:
    problems.append(f"could not check splits: {e}")

def sample_canon(prefix, k=60):
    shards = C.list_shards(REPO_ID, prefix, TOKEN)
    if not shards: return 0, 0
    loc = C.hf_hub_download(REPO_ID, shards[-1], repo_type="dataset", token=TOKEN)
    t   = C.pq.read_table(loc, columns=["image"]); m = t.num_rows
    idx = range(m) if m<=k else random.sample(range(m), k)
    bad = sum(1 for i in idx if not C.png_is_canonical(t.column("image")[i].as_py())[0])
    return len(list(idx)), bad

for p in ["real/"] + [f"data/{m}/" for m in cfg["generators"]]:
    c, bad = sample_canon(p)
    if bad: problems.append(f"{p}: {bad}/{c} not canonical")
    print(f"canonical sample {p}: {c-bad}/{c} ok")

report = {"sniff_test": {k: round(v,3) if v else None for k,v in results.items()},
          "thresholds": {"healthy":"<0.85","investigate":"0.85-0.95","leak":">0.95"},
          "master_seed": MASTER_SEED, "problems": problems}
tmp = os.path.join(tempfile.gettempdir(), "validation_report.json")
json.dump(report, open(tmp,"w"), indent=2)
C.HfApi(token=TOKEN).upload_file(path_or_fileobj=tmp, path_in_repo="validation_report.json",
        repo_id=REPO_ID, repo_type="dataset", commit_message="validation report")
print("\nRESULT:", "PASS" if not problems else "REVIEW PROBLEMS ABOVE")
print(json.dumps(report, indent=2))

data/sd15/sd15-9869edce-00000.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/sd15/sd15-9869edce-00001.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

data/sd15/sd15-9869edce-00002.parquet:   0%|          | 0.00/128M [00:00<?, ?B/s]

data/sd15/sd15-9869edce-00003.parquet:   0%|          | 0.00/127M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


...

canonical sample data/kandinsky22/: 2/2 ok


data/pixart_sigma/pixart_sigma-fcc647dd-(…):   0%|          | 0.00/4.54M [00:00<?, ?B/s]

canonical sample data/pixart_sigma/: 12/12 ok


data/sd_cascade/sd_cascade-fcbce73c-0001(…):   0%|          | 0.00/10.4k [00:00<?, ?B/s]

canonical sample data/sd_cascade/: 7/7 ok

RESULT: PASS
{
  "sniff_test": {
    "sd15": 0.853,
    "sdxl": 0.973,
    "flux_schnell": 0.877,
    "kandinsky22": 0.96,
    "pixart_sigma": 0.932,
    "sd_cascade": 1.0,
    "__overall__": 0.847
  },
  "thresholds": {
    "healthy": "<0.85",
    "investigate": "0.85-0.95",
    "leak": ">0.95"
  },
  "master_seed": 42,
  "problems": []
}
